In [1]:
!python3 -m pip install requests beautifulsoup4


Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.


In [2]:
import requests
import json
import time
import os
from bs4 import BeautifulSoup
import html

In [3]:
def clean_text(text):
    if not text:
        return ""
    
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    
    return text.strip()

In [4]:
def build_query(gene, variant):
    return f'("{gene}" AND "{variant}") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")'

In [5]:
DATA_DIR = "data/literature"
os.makedirs(DATA_DIR, exist_ok=True)

In [6]:
def save_corpus(gene, variant, papers):
    path = f"{DATA_DIR}/{gene}_{variant}_corpus.json"
    
    if not papers:
        print("⚠️ No literature found — saving empty file")
    
    with open(path, "w") as f:
        json.dump(papers, f, indent=4, ensure_ascii=False)
    
    print(f"✅ Saved to {path}")

In [7]:
def fetch_europepmc(query, max_papers=100):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    
    papers = []
    seen = set()
    cursor = "*"
    
    while len(papers) < max_papers:
        params = {
            "query": query,
            "format": "json",
            "pageSize": 50,
            "resultType": "core",
            "cursorMark": cursor
        }
        
        # 🔁 Retry with exponential backoff
        for attempt in range(3):
            try:
                time.sleep(1)  # rate limiting
                
                resp = requests.get(url, params=params, timeout=10)
                data = resp.json()
                break
            
            except Exception as e:
                print(f"⚠️ Retry {attempt+1} failed: {e}")
                time.sleep(2 ** attempt)
        else:
            print("❌ API failed after retries")
            return papers
        
        results = data.get("resultList", {}).get("result", [])
        
        if not results:
            break
        
        for item in results:
            
            # ✅ Get both identifiers
            pmid = item.get("pmid")
            paper_id = item.get("id")
            
            # ❌ Remove preprints (ROBUST FIX 🔥)
            if paper_id and str(paper_id).startswith("PPR"):
                continue
            
            # ✅ Final ID (prefer pmid)
            pmid = pmid or paper_id
            
            # ✅ Clean text (handles HTML + entities)
            title = clean_text(item.get("title", ""))
            abstract = clean_text(item.get("abstractText", ""))
            
            # ✅ Filtering
            if not pmid:
                continue
            if not abstract or len(abstract) < 100:
                continue
            
            # ✅ Deduplication
            if pmid in seen:
                continue
            seen.add(pmid)
            
            # ✅ Store clean schema
            papers.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract
            })
            
            if len(papers) >= max_papers:
                break
        
        # ✅ Pagination
        cursor = data.get("nextCursorMark")
        if not cursor:
            break
    
    print(f"✅ Final valid papers: {len(papers)}")
    return papers

In [8]:
def fetch_literature(gene, variant):
    query = build_query(gene, variant)
    
    print(f"🔍 Query: {query}")
    
    papers = fetch_europepmc(query)
    
    print(f"📄 Papers fetched: {len(papers)}")
    
    # ✅ Check sample cleaned output
    if papers:
        print("\n🔍 SAMPLE CLEAN TITLE:")
        print(papers[0]["title"])
    
    save_corpus(gene, variant, papers)
    
    return papers

In [9]:
papers = fetch_literature("SNCA", "A53T")

🔍 Query: ("SNCA" AND "A53T") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")
✅ Final valid papers: 100
📄 Papers fetched: 100

🔍 SAMPLE CLEAN TITLE:
Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.
✅ Saved to data/literature/SNCA_A53T_corpus.json


In [10]:
papers = fetch_literature("SNCA", "A53T")

papers[:3]

🔍 Query: ("SNCA" AND "A53T") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")
✅ Final valid papers: 100
📄 Papers fetched: 100

🔍 SAMPLE CLEAN TITLE:
Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.
✅ Saved to data/literature/SNCA_A53T_corpus.json


[{'pmid': '41002401',
  'title': 'Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.',
  'abstract': 'Animal and cellular models harboring SNCA gene mutations have been instrumental in synucleinopathy, but faithful human brain models remain limited. Here, we report the development of a human cerebral organoid (CO) model of synucleinopathy carrying the Ala53Thr mutation in SNCA ( SNCA A53T ). Using a human embryonic stem cell (hESC) line overexpressing SNCA A53T (A53T hESC line), we generated COs (A53T COs) that recapitulate hallmark features of synucleinopathy. These A53T COs exhibited elevated α-synuclein (α-Syn) expression, the increased phosphorylation of α-Syn, and Lewy body-like aggregations. Notably, we also observed the increased expression of phosphorylated tau and neurofibrillary tangle-like silver deposits, although amyloid β expression and accumulation remained unchanged. To evaluate the utility of this model in drug screening, we treated A53T COs with synuclean

In [11]:
print("Total papers:", len(papers))

# Check all PMIDs valid
print("All PMIDs valid:", all(p['pmid'] is not None for p in papers))

# Check abstract quality
print("Min abstract length:", min(len(p['abstract']) for p in papers) if papers else 0)

Total papers: 100
All PMIDs valid: True
Min abstract length: 508


In [12]:
for p in papers:
    if "<" in p["title"] or "<" in p["abstract"]:
        print("\n❌ FOUND ISSUE:")
        print("TITLE:", p["title"])
        print("ABSTRACT:", p["abstract"][:200])
        break

In [13]:
papers = fetch_literature("SNCA", "A53T")
papers[0]["title"]

🔍 Query: ("SNCA" AND "A53T") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")
✅ Final valid papers: 100
📄 Papers fetched: 100

🔍 SAMPLE CLEAN TITLE:
Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.
✅ Saved to data/literature/SNCA_A53T_corpus.json


'Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.'

In [14]:
papers[8]["title"]

'From Yeast to Therapeutics: Modeling Neurodegenerative Diseases in Saccharomyces cerevisiae.'

In [15]:
papers[7]

{'pmid': '41438688',
 'title': 'In vitro 3D models of neuron-astrocyte interactions.',
 'abstract': "The pathological processes of neurodegenerative diseases (e.g., Alzheimer's disease, Parkinson's disease, and Amyotrophic Lateral Sclerosis) also include relationships between neuron and glia cells. Conventional two-dimensional (2D) cell cultures have limitations to mimic the microenvironment of cells inside living organisms because of flaws in intercellular relationships investigated using 2D cell cultures. Recent advances have introduced three-dimensional (3D) cell cultures that have the capability to create 3D cellular architecture to mimic advanced platforms for scientific inquiries into neurodegenerative diseases, simulating microenvironments inside living organisms.This review provides a brief overview of the development of in vitro 3D cell culture models of astrocytes and attempts to highlight the role of astrocytes in crucial pathophysiologic events occurring in 3D cultures. Stu

In [16]:
papers[8]

{'pmid': '41277874',
 'title': 'From Yeast to Therapeutics: Modeling Neurodegenerative Diseases in Saccharomyces cerevisiae.',
 'abstract': "Here, we review the use of Saccharomyces cerevisiae as a powerful model organism for studying cellular processes implicated in neurodegenerative disorders, including stress responses, proteostasis impairment, and vesicle trafficking defects. Over the last two decades, baker's yeast models have been developed for complex diseases such as Parkinson's, Alzheimer's, Huntington's, and Amyotrophic lateral sclerosis (ALS). Yeast cells expressing human proteins, such as amyloid-β, α-synuclein, huntingtin, and TDP-43, have become crucial tools for high-throughput drug screening aimed at counteracting disease progression. These yeast models have unveiled key components involved in the metabolism and toxicity of these proteins, enabling the identification of interacting partners and novel factors within each pathway. Importantly, these pathways were subseque

In [17]:
papers = fetch_literature("SNCA", "A53T")

🔍 Query: ("SNCA" AND "A53T") AND (mutation OR pathogenic OR aggregation OR misfolding OR "protein stability")
✅ Final valid papers: 100
📄 Papers fetched: 100

🔍 SAMPLE CLEAN TITLE:
Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.
✅ Saved to data/literature/SNCA_A53T_corpus.json


In [18]:
any(str(p["pmid"]).startswith("PPR") for p in papers)

False

In [19]:
any("<" in p["title"] or "<" in p["abstract"] for p in papers)

False

In [20]:
min(len(p["abstract"]) for p in papers)

508

In [21]:
all(p["pmid"] is not None for p in papers)

True

In [22]:
len(papers) == len(set(p["pmid"] for p in papers))

True

In [23]:
papers[0]

{'pmid': '41002401',
 'title': 'Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.',
 'abstract': 'Animal and cellular models harboring SNCA gene mutations have been instrumental in synucleinopathy, but faithful human brain models remain limited. Here, we report the development of a human cerebral organoid (CO) model of synucleinopathy carrying the Ala53Thr mutation in SNCA ( SNCA A53T ). Using a human embryonic stem cell (hESC) line overexpressing SNCA A53T (A53T hESC line), we generated COs (A53T COs) that recapitulate hallmark features of synucleinopathy. These A53T COs exhibited elevated α-synuclein (α-Syn) expression, the increased phosphorylation of α-Syn, and Lewy body-like aggregations. Notably, we also observed the increased expression of phosphorylated tau and neurofibrillary tangle-like silver deposits, although amyloid β expression and accumulation remained unchanged. To evaluate the utility of this model in drug screening, we treated A53T COs with synuclean D 